In [ ]:
!pip install newsapi-python

from newsapi import NewsApiClient
import pandas as pd
import time
# ============================================
# 1. List of M7 Companies
# ============================================
M7_COMPANIES = [
    "Apple",
    "Microsoft",
    "Amazon",
    "Alphabet",
    "Meta",
    "Tesla",
    "NVIDIA"
]

# ============================================
# 2. Initialize NewsAPI Client
# ============================================
newsapi = NewsApiClient(api_key="4b66a3a7ecae4f63a05b1f03cc720704")

# ============================================
# 3. Query Parameters
# ============================================
FROM_DATE = "2025-11-01"   # Start date
TO_DATE   = "2025-11-24"   # End date
PAGE_SIZE = 100           # Max allowed per request

# List for storing all collected articles
all_rows = []


# ============================================
# 4. Loop Through M7 Companies
# ============================================
for company in M7_COMPANIES:
    print(f"\n Fetching news for: {company}")

    page = 1  # Start from page 1 for each company

    while True:
        # ============================================
        # 4.1 API Request for One Page of Articles
        # ============================================
        response = newsapi.get_everything(
            q=f"{company} OR {company.split()[0]}",   # Use full name + short name
            language='en',
            sort_by="publishedAt",
            from_param=FROM_DATE,
            to=TO_DATE,
            page_size=PAGE_SIZE,
            page=page
        )

        articles = response.get("articles", [])

        # Stop if API returns an empty list → no more pages
        if not articles:
            print(f"   No more articles. Total pages fetched: {page - 1}")
            break

        # ============================================
        # 4.2 Process Each Article
        # ============================================
        for a in articles:
            all_rows.append({
                "title":        a.get("title"),
                "description":  a.get("description"),
                "content":      a.get("content"),
                "published_at": a.get("publishedAt"),
                "source":       a.get("source", {}).get("name"),
                "company":      company    # Label the originating company
            })

        print(f"   Page {page} fetched. Articles: {len(articles)}")

        # Go to the next page
        page += 1

        # Small delay to avoid hitting API rate limits
        time.sleep(0.4)


# ============================================
# 5. Convert to DataFrame
# ============================================
df = pd.DataFrame(all_rows)
print("\nFinal dataset shape:", df.shape)
print(df.head())


# ============================================
# 6. Save to CSV for ETL / Warehouse
# ============================================
df.to_csv("m7_news_202511.csv", index=False, encoding="utf-8")
print("\n Saved to: m7_news_202511.csv")

# ============================================
# 7. Save as JSON (NEW)
# ============================================
json_path = "m7_news_202511.json"

# Save JSON as list of records
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)

print(f"JSON saved to: {json_path}")
